In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog qiskit-serverless
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Écrire ton premier programme Qiskit Serverless

<Accordion>
<AccordionItem title="Package versions">

Le code de cette page a été développé avec les dépendances suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=1.3.1
qiskit-ibm-runtime~=0.34.0
qiskit-aer~=0.15.1
qiskit-serverless~=0.18.1
qiskit-ibm-catalog~=0.2
qiskit-addon-sqd~=0.8.1
qiskit-addon-utils~=0.1.0
qiskit-addon-mpf~=0.2.0
qiskit-addon-aqc-tensor~=0.1.2
qiskit-addon-obp~=0.1.0
scipy~=1.15.0
pyscf~=2.8.0
```
</AccordionItem>
</Accordion>

> **Tip:** **Qiskit Serverless est en cours de mise à niveau et ses fonctionnalités évoluent rapidement.** Pendant cette phase de développement, consulte les notes de version et la documentation la plus récente sur la page [GitHub de Qiskit Serverless](https://qiskit.github.io/qiskit-serverless/index.html).

Cet exemple montre comment utiliser les outils `qiskit-serverless` pour créer un programme de transpilation parallèle, puis utiliser `qiskit-ibm-catalog` pour téléverser ton programme sur IBM Quantum Platform et l'utiliser comme service distant réutilisable.

## Vue d'ensemble du workflow
1. Créer un répertoire local et un fichier de programme vide (`./source_files/transpile_remote.py`)
1. Ajouter du code à ton programme qui, une fois téléversé dans Qiskit Serverless, transpilera un circuit
1. Utiliser `qiskit-ibm-catalog` pour s'authentifier à Qiskit Serverless
1. Téléverser le programme dans Qiskit Serverless

Après avoir téléversé ton programme, tu pourras l'exécuter pour transpiler le circuit en suivant le guide [Exécuter ta première charge de travail Qiskit Serverless à distance](/guides/serverless-run-first-workload).

## Exemple : Transpilation à distance avec Qiskit Serverless
Cet exemple te guide dans la création et l'ajout de code à un fichier de programme qui, une fois téléversé dans Qiskit Serverless, transpilera un `circuit` par rapport à un `backend` donné et un `optimization_level` cible.

:::tip

Qiskit Serverless nécessite de configurer les fichiers `.py` de ta charge de travail dans un répertoire dédié. La structure suivante est un exemple de bonne pratique :

In [ ]:
# This cell is hidden from users, it creates a new folder
from pathlib import Path

Path("./source_files").mkdir(exist_ok=True)

In [ ]:
%%writefile ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this locally in a notebook), running this cell saves to disk rather than executing the code.

from qiskit.transpiler import generate_preset_pass_manager

def transpile_remote(circuit, optimization_level, backend):
    """Transpiles an abstract circuit into an ISA circuit for a given backend."""
    pass_manager = generate_preset_pass_manager(
        optimization_level=optimization_level,
		backend=backend
    )
    isa_circuit = pass_manager.run(circuit)
    return isa_circuit

### Add code to get program arguments

Now add the following code to your program file, which sets up program arguments.

Your initial `transpile_remote.py` has three inputs: `circuits`, `backend_name`, and `optimization_level`. Serverless is currently limited to only accept serializable inputs and outputs. For this reason, you cannot pass in `backend` directly, so use `backend_name` as a string instead.

In [ ]:
%%writefile --append ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this locally in a notebook), running this cell saves to disk rather than executing the code.

from qiskit_serverless import get_arguments, save_result, distribute_task, get

# Get program arguments
arguments = get_arguments()
circuits = arguments.get("circuits")
backend_name = arguments.get("backend_name")
optimization_level = arguments.get("optimization_level")

### Ajouter du code pour récupérer les arguments du programme
Maintenant, ajoute le code suivant à ton fichier de programme, qui configure les arguments du programme.

Ton fichier `transpile_remote.py` initial a trois entrées : `circuits`, `backend_name` et `optimization_level`. Serverless est actuellement limité à n'accepter que des entrées et sorties sérialisables. Pour cette raison, tu ne peux pas passer `backend` directement, alors utilise `backend_name` comme chaîne de caractères à la place.

In [ ]:
%%writefile --append ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this locally in a notebook), running this cell saves to disk rather than executing the code.

from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend(backend_name)

### Ajouter du code qui appelle le backend
Ajoute le code suivant à ton fichier de programme, qui appelle ton backend avec `QiskitRuntimeService`.

Le code suivant suppose que tu as déjà suivi le processus pour enregistrer tes identifiants en utilisant `QiskitRuntimeService.save_account`, et chargera ton compte par défaut enregistré sauf indication contraire. Consulte [Enregistrer tes identifiants de connexion](/guides/save-credentials) et [Initialiser ton compte de service Qiskit Runtime](/guides/initialize-account) pour plus d'informations.

In [ ]:
%%writefile --append ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this locally in a notebook), running this cell saves to disk rather than executing the code.

# Each circuit is being transpiled and will populate the array
results = [
    transpile_remote(circuit, 1, backend)
    for circuit in circuits
]

save_result({
    "transpiled_circuits": results
})

### Ajouter du code pour transpiler
Enfin, ajoute le code suivant à ton fichier de programme. Ce code exécute `transpile_remote()` sur tous les `circuits` passés en entrée, et retourne les `transpiled_circuits` comme résultat :

In [13]:
from qiskit_ibm_catalog import QiskitServerless, QiskitFunction

# Authenticate to the remote cluster and submit the pattern for remote execution
serverless = QiskitServerless()

<span id="upload-to-qiskit-serverless"></span>

### S'authentifier à Qiskit Serverless
Utilise `qiskit-ibm-catalog` pour t'authentifier à `QiskitServerless` avec ta clé API (tu peux utiliser ta clé API `QiskitRuntimeService` ou en créer une nouvelle sur le [tableau de bord IBM Quantum Platform](https://quantum.cloud.ibm.com)).

In [8]:
transpile_remote_demo = QiskitFunction(
    title="transpile_remote_serverless",
    entrypoint="transpile_remote.py",
    working_dir="./source_files/",
)

In [9]:
serverless.upload(transpile_remote_demo)

QiskitFunction(transpile_remote_serverless)

### Verify upload

To check if it successfully uploaded, use `serverless.list()`, as in the following code:

In [10]:
# Get program from serverless.list() that matches the title of the one we uploaded
next(
    program
    for program in serverless.list()
    if program.title == "transpile_remote_serverless"
)

QiskitFunction(transpile_remote_serverless)

In [11]:
# This cell is hidden from users, it checks the program uploaded correctly
assert _.title == "transpile_remote_serverless"  # noqa: F821

In [12]:
# This cell is hidden from users, it checks the program executes correctly
from time import sleep
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
transpile_remote_serverless = serverless.load("transpile_remote_serverless")
job = transpile_remote_serverless.run(
    circuits=[qc],
    backend="ibm_sherbrooke",
    optimization_level=1,
)
while True:
    sleep(5)
    status = job.status()
    if status not in ["QUEUED", "INITIALIZING", "RUNNING", "DONE"]:
        raise Exception(
            f"Unexpected job status: '{status}'\n"
            + "Here are the logs:\n"
            + job.logs()
        )
    if status == "DONE":
        break

## Next steps

<Admonition type="info" title="Recommendations">

- Learn how to pass inputs and run your program remotely in the [Run your first Qiskit Serverless workload remotely](/docs/guides/serverless-run-first-workload) topic.

</Admonition>